<a href="https://colab.research.google.com/github/MasterSaha/AI-and-ML-Projects/blob/main/Deep_Neural_Cryptography_SHA_1_Hash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

class PreciseNeuralXOR(nn.Module):
    """
    Implements a perfectly precise 2-input XOR gate using continuous
    linear layers and ReLU activation functions, exactly as outlined
    in the Deep Neural Cryptography framework.
    """
    def __init__(self):
        super(PreciseNeuralXOR, self).__init__()
        # Layer 1: Detects directional differences between the two inputs
        self.layer1 = nn.Linear(2, 2, bias=False)
        with torch.no_grad():
            # Neuron 1 computes: x1 - x2
            # Neuron 2 computes: x2 - x1
            self.layer1.weight.copy_(torch.tensor([
                [1.0, -1.0],
                [-1.0, 1.0]
            ]))

        # Layer 2: Sums up the rectified differences
        self.layer2 = nn.Linear(2, 1, bias=False)
        with torch.no_grad():
            self.layer2.weight.copy_(torch.tensor([[1.0, 1.0]]))

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x


class SanitizationLayer(nn.Module):
    """
    The defensive mechanism proposed to thwart linear-time attacks.
    It snaps continuous, fractional inputs strictly back to the Boolean
    hypercube (0 or 1) before they reach the cryptographic inner layers.
    """
    def __init__(self, threshold=0.5):
        super(SanitizationLayer, self).__init__()
        self.threshold = threshold

    def forward(self, x):
        # In a provably secure structural DNN, this forces non-standard continuous
        # inputs back to solid 0.0 or 1.0 bounds using step-like ReLU combinations.
        return torch.where(x >= self.threshold, torch.tensor(1.0), torch.tensor(0.0))


class NeuralCryptoSystem(nn.Module):
    def __init__(self, protected=False):
        super(NeuralCryptoSystem, self).__init__()
        self.protected = protected
        self.sanitizer = SanitizationLayer()
        self.xor_logic = PreciseNeuralXOR()

    def forward(self, x):
        if self.protected:
            x = self.sanitizer(x)
        return self.xor_logic(x)


# --- Demonstration of the Vulnerability and the Defense ---
if __name__ == "__main__":
    print("=== Deep Neural Cryptography Simulation ===")

    # Standard Boolean inputs work perfectly on both
    standard_input = torch.tensor([[1.0, 0.0]])

    unprotected_net = NeuralCryptoSystem(protected=False)
    protected_net = NeuralCryptoSystem(protected=True)

    print(f"Standard Input [1.0, 0.0] -> Unprotected Output: {unprotected_net(standard_input).item()}")
    print(f"Standard Input [1.0, 0.0] -> Protected Output:   {protected_net(standard_input).item()}\n")

    # The Vulnerability: Attacker injects a non-standard fractional "jitter" input
    # In a broader system like SHA-1 or AES, observing how these continuous variations
    # scale at the output allows solving the internal states in linear time.
    fractional_input = torch.tensor([[0.7, 0.0]])

    print("--- Attacker Injects Non-Standard Input [0.7, 0.0] ---")

    # The unprotected network processes the fractional number continuously, leaking details
    unprotected_output = unprotected_net(fractional_input).item()
    print(f"Unprotected Output: {unprotected_output:.2f} (Leaks the fractional difference linear gradient!)")

    # The protected network sanitizes the input back to the hypercube, neutralizing the attack
    protected_output = protected_net(fractional_input).item()
    print(f"Protected Output:   {protected_output:.2f} (Successfully blocked the continuous exploitation)")

=== Deep Neural Cryptography Simulation ===
Standard Input [1.0, 0.0] -> Unprotected Output: 1.0
Standard Input [1.0, 0.0] -> Protected Output:   1.0

--- Attacker Injects Non-Standard Input [0.7, 0.0] ---
Unprotected Output: 0.70 (Leaks the fractional difference linear gradient!)
Protected Output:   1.00 (Successfully blocked the continuous exploitation)


Test validation - One-block message 24 Bits

In [7]:
import torch
import torch.nn as nn
import struct

class NeuralBitwiseXOR(nn.Module):
    """
    Computes precise bitwise XOR across tensors using continuous ReLU logic.
    Supports a list of multiple tensors to perform n-way XOR.
    """
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()

    def forward(self, tensors):
        # Base case: XORing a single tensor returns itself
        res = tensors[0]
        for next_tensor in tensors[1:]:
            # Core paper mapping: XOR(x, y) = ReLU(x - y) + ReLU(y - x)
            res = self.relu(res - next_tensor) + self.relu(next_tensor - res)
        return res


class NeuralSHA1Round(nn.Module):
    """
    Simulates a single step/round of SHA-1 state transitions using neural building blocks.
    Translates bit-level manipulation, logical combinations, and modular additions.
    """
    def __init__(self):
        super().__init__()
        self.nxor = NeuralBitwiseXOR()
        self.relu = nn.ReLU()

    def _neural_and(self, x, y):
        # Mapping: AND(x, y) = ReLU(x + y - 1)
        return self.relu(x + y - 1.0)

    def _neural_not(self, x):
        # Mapping: NOT(x) = 1 - x
        return 1.0 - x

    def forward(self, A, B, C, D, E, W_t, K_t, round_idx):
        # 1. Evaluate nonlinear round functions (Ch, Parity, Maj) using Neural Gates
        if 0 <= round_idx <= 19:
            # Ch(B, C, D) = (B AND C) XOR (NOT B AND D)
            f = self.nxor([self._neural_and(B, C), self._neural_and(self._neural_not(B), D)])
        elif 20 <= round_idx <= 39 or 60 <= round_idx <= 79:
            # Parity(B, C, D) = B XOR C XOR D
            f = self.nxor([B, C, D])
        elif 40 <= round_idx <= 59:
            # Maj(B, C, D) = (B AND C) XOR (B AND D) XOR (C AND D)
            f = self.nxor([self._neural_and(B, C), self._neural_and(B, D), self._neural_and(C, D)])

        # 2. Simulate Left Rotation (ROTL_5(A))
        # In a structural network, this is a fixed permutation matrix mapping input indexes.
        A_rot5 = torch.roll(A, shifts=-5, dims=-1)

        # 3. Simulate 32-bit Modular Addition using Neural Arithmetic
        # To maintain the neural structural abstraction without leaving the tensor matrix,
        # we convert the continuous outputs back to integers to simulate structural word updates.
        def tensor_to_int(t):
            bits = t.round().long().tolist()[0]
            val = 0
            for b in bits:
                val = (val << 1) | b
            return val

        def int_to_tensor(val):
            bits = [(val >> i) & 1 for i in reversed(range(32))]
            return torch.tensor([bits], dtype=torch.float32)

        # Compute: Temp = ROTL_5(A) + f + E + W_t + K_t
        sum_val = (tensor_to_int(A_rot5) + tensor_to_int(f) +
                   tensor_to_int(E) + tensor_to_int(W_t) + tensor_to_int(K_t)) & 0xFFFFFFFF
        temp = int_to_tensor(sum_val)

        # Shift States safely downstream
        E = D
        D = C
        C = torch.roll(B, shifts=-30, dims=-1) # ROTL_30(B)
        B = A
        A = temp

        return A, B, C, D, E


class FullNeuralSHA1(nn.Module):
    """
    Assembles individual neural step architectures into a full 80-round SHA-1 system.
    Includes the Sanitization layer to neutralize continuous adversarial inputs.
    """
    def __init__(self, protected=True):
        super().__init__()
        self.protected = protected
        self.round_layer = NeuralSHA1Round()
        self.nxor = NeuralBitwiseXOR()

        # Constant words defined by SHA-1 specification mapped into tensor representations
        self.K = [
            self._int_to_tensor(0x5A827999), # Rounds 0-19
            self._int_to_tensor(0x6ED9EBA1), # Rounds 20-39
            self._int_to_tensor(0x8F1BBCDC), # Rounds 40-59
            self._int_to_tensor(0xCA62C1D6)  # Rounds 60-79
        ]

    def _int_to_tensor(self, val):
        bits = [(val >> i) & 1 for i in reversed(range(32))]
        return torch.tensor([bits], dtype=torch.float32)

    def _sanitize(self, x):
        """Snaps continuous adversarial jitters strictly back to binary values"""
        return torch.where(x >= 0.5, torch.tensor(1.0), torch.tensor(0.0))

    def forward(self, msg_bits):
        # 1. Enforce Sanitization Boundary Layer if active
        if self.protected:
            msg_bits = self._sanitize(msg_bits)

        # 2. Process Message Schedule (W) Neurally
        # Reshape input bit vector into sixteen 32-bit words
        W = [msg_bits[:, i*32:(i+1)*32] for i in range(16)]

        # Expand message schedule out to 80 words using continuous neural XOR operations
        for t in range(16, 80):
            # W[t] = ROTL_1(W[t-3] XOR W[t-8] XOR W[t-14] XOR W[t-16])
            xor_out = self.nxor([W[t-3], W[t-8], W[t-14], W[t-16]])
            W_t = torch.roll(xor_out, shifts=-1, dims=-1)
            W.append(W_t)

        # 3. Initialize standard SHA-1 H-registers
        H0 = self._int_to_tensor(0x67452301)
        H1 = self._int_to_tensor(0xEFCDAB89)
        H2 = self._int_to_tensor(0x98BADCFE)
        H3 = self._int_to_tensor(0x10325476)
        H4 = self._int_to_tensor(0xC3D2E1F0)

        A, B, C, D, E = H0, H1, H2, H3, H4

        # 4. Execute the 80 sequential neural rounds
        for t in range(80):
            K_t = self.K[t // 20]
            A, B, C, D, E = self.round_layer(A, B, C, D, E, W[t], K_t, t)

        # Helpers to resolve final addition
        def t_to_i(t):
            return int("".join([str(int(b)) for b in t.round().tolist()[0]]), 2)

        # Final Hash computation adding registers back to initial states
        h0_final = (t_to_i(A) + t_to_i(H0)) & 0xFFFFFFFF
        h1_final = (t_to_i(B) + t_to_i(H1)) & 0xFFFFFFFF
        h2_final = (t_to_i(C) + t_to_i(H2)) & 0xFFFFFFFF
        h3_final = (t_to_i(D) + t_to_i(H3)) & 0xFFFFFFFF
        h4_final = (t_to_i(E) + t_to_i(H4)) & 0xFFFFFFFF

        # Render output digest string
        return f"{h0_final:08x} {h1_final:08x} {h2_final:08x} {h3_final:08x} {h4_final:08x}"


# --- Verification Pipeline ---
if __name__ == "__main__":
    # Test Input Setup: "abc"
    message_bytes = b"abc"

    # Apply standard SHA-1 structural padding rules for single block execution (512 bits)
    bit_len = len(message_bytes) * 8
    padded_msg = message_bytes + b"\x80"
    padded_msg += b"\x00" * (56 - len(padded_msg))
    padded_msg += struct.pack(">Q", bit_len) # Appends original bit length as 64-bit Big-Endian int

    # Convert the full 512-bit message block into a binary vector tensor
    msg_bits_list = []
    for byte in padded_msg:
        msg_bits_list.extend([(byte >> i) & 1 for i in reversed(range(8))])

    input_tensor = torch.tensor([msg_bits_list], dtype=torch.float32)

    # Instantiate and validate the Neural Implementation
    sha1_neural_net = FullNeuralSHA1(protected=True)
    hash_output = sha1_neural_net(input_tensor)

    print("--- SHA-1 Neural Verification Summary ---")
    print(f"Target Output: a9993e36 4706816a ba3e2571 7850c26c 9cd0d89d")
    print(f"Neural Output: {hash_output}")
    print(f"Match Confirmed: {hash_output == 'a9993e36 4706816a ba3e2571 7850c26c 9cd0d89d'}")

--- SHA-1 Neural Verification Summary ---
Target Output: a9993e36 4706816a ba3e2571 7850c26c 9cd0d89d
Neural Output: a9993e36 4706816a ba3e2571 7850c26c 9cd0d89d
Match Confirmed: True


Test validation - Multi-block message 448 bits

In [11]:
import torch
import torch.nn as nn
import struct

class NeuralBitwiseXOR(nn.Module):
    """
    Computes precise bitwise XOR across tensors using continuous ReLU logic.
    Supports a list of multiple tensors to perform n-way XOR.
    """
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()

    def forward(self, tensors):
        res = tensors[0]
        for next_tensor in tensors[1:]:
            res = self.relu(res - next_tensor) + self.relu(next_tensor - res)
        return res


class NeuralSHA1Round(nn.Module):
    """
    Simulates a single step/round of SHA-1 state transitions using neural building blocks.
    Translates bit-level manipulation, logical combinations, and modular additions.
    """
    def __init__(self):
        super().__init__()
        self.nxor = NeuralBitwiseXOR()
        self.relu = nn.ReLU()

    def _neural_and(self, x, y):
        return self.relu(x + y - 1.0)

    def _neural_not(self, x):
        return 1.0 - x

    def forward(self, A, B, C, D, E, W_t, K_t, round_idx):
        # 1. Evaluate nonlinear round functions using Neural Gates
        if 0 <= round_idx <= 19:
            f = self.nxor([self._neural_and(B, C), self._neural_and(self._neural_not(B), D)])
        elif 20 <= round_idx <= 39 or 60 <= round_idx <= 79:
            f = self.nxor([B, C, D])
        elif 40 <= round_idx <= 59:
            f = self.nxor([self._neural_and(B, C), self._neural_and(B, D), self._neural_and(C, D)])

        # 2. Simulate Left Rotation (ROTL_5(A))
        A_rot5 = torch.roll(A, shifts=-5, dims=-1)

        # 3. Simulate 32-bit Modular Addition using Neural Arithmetic
        def tensor_to_int(t):
            bits = t.round().long().tolist()[0]
            val = 0
            for b in bits:
                val = (val << 1) | b
            return val

        def int_to_tensor(val):
            bits = [(val >> i) & 1 for i in reversed(range(32))]
            return torch.tensor([bits], dtype=torch.float32)

        sum_val = (tensor_to_int(A_rot5) + tensor_to_int(f) +
                   tensor_to_int(E) + tensor_to_int(W_t) + tensor_to_int(K_t)) & 0xFFFFFFFF
        temp = int_to_tensor(sum_val)

        # Shift States safely downstream
        E = D
        D = C
        C = torch.roll(B, shifts=-30, dims=-1) # ROTL_30(B)
        B = A
        A = temp

        return A, B, C, D, E


class FullNeuralSHA1(nn.Module):
    """
    Assembles structural neural layers to compute SHA-1 over multiple 512-bit blocks.
    Includes a Sanitization Layer to neutralize continuous adversarial vulnerabilities.
    """
    def __init__(self, protected=True):
        super().__init__()
        self.protected = protected
        self.round_layer = NeuralSHA1Round()
        self.nxor = NeuralBitwiseXOR()

        # Constant words defined by SHA-1 specification mapped into tensor representations
        self.K = [
            self._int_to_tensor(0x5A827999), # Rounds 0-19
            self._int_to_tensor(0x6ED9EBA1), # Rounds 20-39
            self._int_to_tensor(0x8F1BBCDC), # Rounds 40-59
            self._int_to_tensor(0xCA62C1D6)  # Rounds 60-79
        ]

    def _int_to_tensor(self, val):
        bits = [(val >> i) & 1 for i in reversed(range(32))]
        return torch.tensor([bits], dtype=torch.float32)

    def _sanitize(self, x):
        """Snaps continuous adversarial jitters strictly back to binary values"""
        return torch.where(x >= 0.5, torch.tensor(1.0), torch.tensor(0.0))

    def _tensor_to_int(self, t):
        return int("".join([str(int(b)) for b in t.round().tolist()[0]]), 2)

    def forward(self, msg_bits_all_blocks):
        # 1. Enforce Sanitization Boundary Layer if active
        if self.protected:
            msg_bits_all_blocks = self._sanitize(msg_bits_all_blocks)

        # 2. Initialize standard SHA-1 H-registers
        H0 = self._int_to_tensor(0x67452301)
        H1 = self._int_to_tensor(0xEFCDAB89)
        H2 = self._int_to_tensor(0x98BADCFE)
        H3 = self._int_to_tensor(0x10325476)
        H4 = self._int_to_tensor(0xC3D2E1F0)

        # Calculate the total number of 512-bit blocks provided
        num_blocks = msg_bits_all_blocks.shape[1] // 512

        # 3. Loop sequentially through each 512-bit block
        for block_idx in range(num_blocks):
            block_bits = msg_bits_all_blocks[:, block_idx*512:(block_idx+1)*512]

            # Process Message Schedule (W) Neurally for current block
            W = [block_bits[:, i*32:(i+1)*32] for i in range(16)]
            for t in range(16, 80):
                xor_out = self.nxor([W[t-3], W[t-8], W[t-14], W[t-16]])
                W_t = torch.roll(xor_out, shifts=-1, dims=-1)
                W.append(W_t)

            # Working registers start as the current state of the H-registers
            A, B, C, D, E = H0, H1, H2, H3, H4

            # Execute the 80 sequential neural rounds for this block
            for t in range(80):
                K_t = self.K[t // 20]
                A, B, C, D, E = self.round_layer(A, B, C, D, E, W[t], K_t, t)

            # Update H-registers for the next iteration using modular addition
            h0_int = (self._tensor_to_int(A) + self._tensor_to_int(H0)) & 0xFFFFFFFF
            h1_int = (self._tensor_to_int(B) + self._tensor_to_int(H1)) & 0xFFFFFFFF
            h2_int = (self._tensor_to_int(C) + self._tensor_to_int(H2)) & 0xFFFFFFFF
            h3_int = (self._tensor_to_int(D) + self._tensor_to_int(H3)) & 0xFFFFFFFF
            h4_int = (self._tensor_to_int(E) + self._tensor_to_int(H4)) & 0xFFFFFFFF

            H0 = self._int_to_tensor(h0_int)
            H1 = self._int_to_tensor(h1_int)
            H2 = self._int_to_tensor(h2_int)
            H3 = self._int_to_tensor(h3_int)
            H4 = self._int_to_tensor(h4_int)

        # Render final output digest string from the terminal state values
        return f"{self._tensor_to_int(H0):08x} {self._tensor_to_int(H1):08x} {self._tensor_to_int(H2):08x} {self._tensor_to_int(H3):08x} {self._tensor_to_int(H4):08x}"


# --- Multi-Block Verification Pipeline ---
if __name__ == "__main__":
    # Test Input Setup (448 bits of standard ASCII string text)
    message_bytes = b"abcdbcdecdefdefgefghfghighijhijkijkljklmklmlmnmnnmnnopop"

    # Apply standard SHA-1 structural padding rules
    bit_len = len(message_bytes) * 8
    padded_msg = message_bytes + b"\x80"

    # Calculate padding bytes needed to make final size a multiple of 512 bits
    # The last 8 bytes are strictly reserved for the big-endian 64-bit length descriptor
    mod_rem = (len(padded_msg) + 8) % 64
    if mod_rem != 0:
        padded_msg += b"\x00" * (64 - mod_rem)
    else:
        # If perfectly aligned, it still needs to fill out to the next block boundary
        pass

    # Finalize the padding configuration with the bit length descriptor
    padded_msg += struct.pack(">Q", bit_len)

    # Convert the multi-block padded message string into a flat binary vector tensor
    msg_bits_list = []
    for byte in padded_msg:
        msg_bits_list.extend([(byte >> i) & 1 for i in reversed(range(8))])

    input_tensor = torch.tensor([msg_bits_list], dtype=torch.float32)

    # Instantiate and validate the Multi-Block Neural Implementation
    sha1_neural_net = FullNeuralSHA1(protected=True)
    hash_output = sha1_neural_net(input_tensor)

    print("--- SHA-1 Multi-Block Neural Verification Summary ---")
    print(f"Total Blocks Processed: {input_tensor.shape[1] // 512}")
    print(f"Target Output:           abbe3d9e 1399de1c 8ef89709 0e2e57d5 d234bad3")
    print(f"Neural Output:           {hash_output}")
    print(f"Match Confirmed:         {hash_output == 'abbe3d9e 1399de1c 8ef89709 0e2e57d5 d234bad3'}")

--- SHA-1 Multi-Block Neural Verification Summary ---
Total Blocks Processed: 2
Target Output:           abbe3d9e 1399de1c 8ef89709 0e2e57d5 d234bad3
Neural Output:           abbe3d9e 1399de1c 8ef89709 0e2e57d5 d234bad3
Match Confirmed:         True


Test validation - Long message - 1,000,000 repetitions of the character 'a'

In [16]:
### PyTorch Optimized Long-Message Neural SHA-1 Implementation

import torch
import torch.nn as nn
import struct

class NeuralBitwiseXOR(nn.Module):
    """
    Computes precise bitwise XOR across tensors using continuous ReLU logic.
    Supports a list of multiple tensors to perform n-way XOR.
    """
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()

    def forward(self, tensors):
        res = tensors[0]
        for next_tensor in tensors[1:]:
            res = self.relu(res - next_tensor) + self.relu(next_tensor - res)
        return res


class NeuralSHA1Round(nn.Module):
    """
    Simulates a single step/round of SHA-1 state transitions using neural building blocks.
    Translates bit-level manipulation, logical combinations, and modular additions.
    """
    def __init__(self):
        super().__init__()
        self.nxor = NeuralBitwiseXOR()
        self.relu = nn.ReLU()

    def _neural_and(self, x, y):
        return self.relu(x + y - 1.0)

    def _neural_not(self, x):
        return 1.0 - x

    def forward(self, A, B, C, D, E, W_t, K_t, round_idx):
        # 1. Evaluate nonlinear round functions using Neural Gates
        if 0 <= round_idx <= 19:
            f = self.nxor([self._neural_and(B, C), self._neural_and(self._neural_not(B), D)])
        elif 20 <= round_idx <= 39 or 60 <= round_idx <= 79:
            f = self.nxor([B, C, D])
        elif 40 <= round_idx <= 59:
            f = self.nxor([self._neural_and(B, C), self._neural_and(B, D), self._neural_and(C, D)])

        # 2. Simulate Left Rotation (ROTL_5(A))
        A_rot5 = torch.roll(A, shifts=-5, dims=-1)

        # 3. Simulate 32-bit Modular Addition using Neural Arithmetic
        def tensor_to_int(t):
            bits = t.round().long().tolist()[0]
            val = 0
            for b in bits:
                val = (val << 1) | b
            return val

        def int_to_tensor(val):
            bits = [(val >> i) & 1 for i in reversed(range(32))]
            return torch.tensor([bits], dtype=torch.float32)

        sum_val = (tensor_to_int(A_rot5) + tensor_to_int(f) +
                   tensor_to_int(E) + tensor_to_int(W_t) + tensor_to_int(K_t)) & 0xFFFFFFFF
        temp = int_to_tensor(sum_val)

        # Shift States safely downstream
        E = D
        D = C
        C = torch.roll(B, shifts=-30, dims=-1) # ROTL_30(B)
        B = A
        A = temp

        return A, B, C, D, E


class FullNeuralSHA1(nn.Module):
    """
    Assembles structural neural layers to compute SHA-1 over streamed 512-bit blocks.
    Optimized to run inside a no_grad environment to process long data pipelines safely.
    """
    def __init__(self, protected=True):
        super().__init__()
        self.protected = protected
        self.round_layer = NeuralSHA1Round()
        self.nxor = NeuralBitwiseXOR()

        # Constant words defined by SHA-1 specification mapped into tensor representations
        self.K = [
            self._int_to_tensor(0x5A827999), # Rounds 0-19
            self._int_to_tensor(0x6ED9EBA1), # Rounds 20-39
            self._int_to_tensor(0x8F1BBCDC), # Rounds 40-59
            self._int_to_tensor(0xCA62C1D6)  # Rounds 60-79
        ]

    def _int_to_tensor(self, val):
        bits = [(val >> i) & 1 for i in reversed(range(32))]
        return torch.tensor([bits], dtype=torch.float32)

    def _sanitize(self, x):
        """Snaps continuous adversarial jitters strictly back to binary values"""
        return torch.where(x >= 0.5, torch.tensor(1.0), torch.tensor(0.0))

    def _tensor_to_int(self, t):
        return int("".join([str(int(b)) for b in t.round().tolist()[0]]), 2)

    def process_stream(self, block_iterator):
        """
        Accepts a generator yielding individual 512-bit block tensors.
        Maintains flat O(1) memory complexity throughout execution.
        """
        # Initialize standard SHA-1 H-registers
        H0 = self._int_to_tensor(0x67452301)
        H1 = self._int_to_tensor(0xEFCDAB89)
        H2 = self._int_to_tensor(0x98BADCFE)
        H3 = self._int_to_tensor(0x10325476)
        H4 = self._int_to_tensor(0xC3D2E1F0)

        for block_bits in block_iterator:
            if self.protected:
                block_bits = self._sanitize(block_bits)

            # Process Message Schedule (W) Neurally for the current block
            W = [block_bits[:, i*32:(i+1)*32] for i in range(16)]
            for t in range(16, 80):
                xor_out = self.nxor([W[t-3], W[t-8], W[t-14], W[t-16]])
                W_t = torch.roll(xor_out, shifts=-1, dims=-1)
                W.append(W_t)

            # Working registers start as the current state of the H-registers
            A, B, C, D, E = H0, H1, H2, H3, H4

            # Execute the 80 sequential neural rounds for this block
            for t in range(80):
                K_t = self.K[t // 20]
                A, B, C, D, E = self.round_layer(A, B, C, D, E, W[t], K_t, t)

            # Update H-registers for the next iteration using modular addition
            h0_int = (self._tensor_to_int(A) + self._tensor_to_int(H0)) & 0xFFFFFFFF
            h1_int = (self._tensor_to_int(B) + self._tensor_to_int(H1)) & 0xFFFFFFFF
            h2_int = (self._tensor_to_int(C) + self._tensor_to_int(H2)) & 0xFFFFFFFF
            h3_int = (self._tensor_to_int(D) + self._tensor_to_int(H3)) & 0xFFFFFFFF
            h4_int = (self._tensor_to_int(E) + self._tensor_to_int(H4)) & 0xFFFFFFFF

            H0 = self._int_to_tensor(h0_int)
            H1 = self._int_to_tensor(h1_int)
            H2 = self._int_to_tensor(h2_int)
            H3 = self._int_to_tensor(h3_int)
            H4 = self._int_to_tensor(h4_int)

        # Render final output digest string from the terminal state values
        return f"{self._tensor_to_int(H0):08x} {self._tensor_to_int(H1):08x} {self._tensor_to_int(H2):08x} {self._tensor_to_int(H3):08x} {self._tensor_to_int(H4):08x}"


# --- Streaming Block Generator ---
def bytes_to_bits_tensor(byte_data):
    bits = []
    for byte in byte_data:
        bits.extend([(byte >> i) & 1 for i in reversed(range(8))])
    return torch.tensor([bits], dtype=torch.float32)

def sha1_million_a_generator():
    """
    Generator function that simulates and streams the structural padding steps
    of 1,000,000 'a' characters without materializing the full array in memory.
    """
    total_chars = 1_000_000
    total_bits = total_chars * 8

    # 1,000,000 'a' characters equal 1,000,000 bytes.
    # 1,000,000 / 64 = 15625 full blocks exactly.
    # Because it hits a block boundary exactly, the padding (\x80 + zeroes + length descriptor)
    # spills entirely into a brand new 15,626th block.

    # Stream the first 15,625 pristine blocks of pure 'a's (0x61)
    pure_block = b'a' * 64
    pure_tensor = bytes_to_bits_tensor(pure_block)
    for _ in range(15625):
        yield pure_tensor

    # Construct the final 15,626th block containing the structural padding bits
    final_block_bytes = b"\x80" + (b"\x00" * 55) + struct.pack(">Q", total_bits)
    yield bytes_to_bits_tensor(final_block_bytes)


# --- Long-Message Verification Pipeline ---
if __name__ == "__main__":
    print("--- Starting Long Message Neural SHA-1 Pipeline ---")
    print("Processing 1,000,000 'a' repetitions...")

    # Initialize the architecture
    sha1_neural_net = FullNeuralSHA1(protected=True)

    # Process the stream completely within no_grad context to ensure O(1) memory scaling
    with torch.no_grad():
        block_stream = sha1_million_a_generator()
        hash_output = sha1_neural_net.process_stream(block_stream)

    print("\n--- SHA-1 Long-Message Neural Verification Summary ---")
    print(f"Target Output: 34aa973c d4c4daa4 f61eeb2b dbad2731 6534016f")
    print(f"Neural Output: {hash_output}")
    print(f"Match Confirmed: {hash_output == '34aa973c d4c4daa4 f61eeb2b dbad2731 6534016f'}")


--- Starting Long Message Neural SHA-1 Pipeline ---
Processing 1,000,000 'a' repetitions...

--- SHA-1 Long-Message Neural Verification Summary ---
Target Output: 34aa973c d4c4daa4 f61eeb2b dbad2731 6534016f
Neural Output: 34aa973c d4c4daa4 f61eeb2b dbad2731 6534016f
Match Confirmed: True
